In [9]:
import pandas as pd
import numpy as np

In [10]:
# Membaca sheet sales
sales = pd.read_excel(
    "FIX_RAW.xlsx",
    sheet_name="sales"
)

# Membaca daftar produk
with open("fix_data_produk.txt", "r", encoding="utf-8") as f:
    daftar_produk = [
        line.strip()
        for line in f
        if line.strip()
    ]

print("Jumlah data sales:", len(sales))
print("Jumlah produk dalam TXT:", len(daftar_produk))

Jumlah data sales: 3211
Jumlah produk dalam TXT: 45


In [11]:
print(sales.columns.tolist())

['tanggal', 'qty', 'nama_produk']


### Cek tanggal terlewat

In [12]:
# Pastikan kolom tanggal bertipe datetime
sales["tanggal"] = pd.to_datetime(sales["tanggal"], errors="coerce")

# Periode yang seharusnya
tanggal_mulai = pd.Timestamp("2022-01-01")
tanggal_akhir = pd.Timestamp("2024-12-31")

# Membuat seluruh tanggal yang seharusnya ada
semua_tanggal = pd.date_range(
    start=tanggal_mulai,
    end=tanggal_akhir,
    freq="D"
)

# Ambil tanggal yang benar-benar ada di sales
tanggal_sales = sales["tanggal"].dropna().dt.normalize().unique()

# Cari tanggal yang tidak ada
tanggal_terlewat = semua_tanggal[
    ~semua_tanggal.isin(tanggal_sales)
]

print("Jumlah tanggal yang seharusnya :", len(semua_tanggal))
print("Jumlah tanggal yang tersedia   :", len(tanggal_sales))
print("Jumlah tanggal terlewat        :", len(tanggal_terlewat))

Jumlah tanggal yang seharusnya : 1096
Jumlah tanggal yang tersedia   : 552
Jumlah tanggal terlewat        : 544


In [13]:
hasil_tanggal = pd.DataFrame({
    "tanggal_terlewat": tanggal_terlewat
})

display(hasil_tanggal)

,tanggal_terlewat
0,2022-01-02
1,2022-01-15
2,2022-02-08
3,2022-02-13
4,2022-02-18
...,...
539,2024-12-27
540,2024-12-28
541,2024-12-29
542,2024-12-30


### Cek produk duplikat pada tanggal yang sama

In [14]:
duplikat = sales[
    sales.duplicated(
        subset=["tanggal", "nama_produk"],
        keep=False
    )
].copy()

duplikat = duplikat.sort_values(
    ["tanggal", "nama_produk"]
)

print("Jumlah baris yang terlibat dalam duplikasi:",
      len(duplikat))

display(duplikat)

Jumlah baris yang terlibat dalam duplikasi: 770


,tanggal,qty,nama_produk
1,2022-01-01,7,media
8,2022-01-01,2,media
22,2022-01-03,16,brokoli_kuning
27,2022-01-03,24,brokoli_kuning
15,2022-01-03,1,media
...,...,...,...
3175,2024-11-22,15,media
3181,2024-11-24,5,media
3186,2024-11-24,1,media
3209,2024-12-04,1,media


In [15]:
rekap_duplikat = (
    sales
    .groupby(["tanggal", "nama_produk"])
    .size()
    .reset_index(name="jumlah")
)

rekap_duplikat = rekap_duplikat[
    rekap_duplikat["jumlah"] > 1
]

rekap_duplikat = rekap_duplikat.sort_values(
    ["tanggal", "nama_produk"]
)

print(
    "Jumlah kombinasi tanggal + produk yang duplikat:",
    len(rekap_duplikat)
)

display(rekap_duplikat)

Jumlah kombinasi tanggal + produk yang duplikat: 326


,tanggal,nama_produk,jumlah
4,2022-01-01,media,2
12,2022-01-03,brokoli_kuning,2
14,2022-01-03,media,4
16,2022-01-03,melati_mini,2
22,2022-01-04,media,3
...,...,...,...
2713,2024-11-17,cagak_pot,2
2718,2024-11-17,media,4
2734,2024-11-22,media,4
2740,2024-11-24,media,2


### Cek nama produk yang tidak ada di daftar produk

In [16]:
# Pastikan nama produk berupa string
sales["nama_produk"] = sales["nama_produk"].astype(str).str.strip()

# Bersihkan daftar produk TXT
daftar_produk = [
    produk.strip()
    for produk in daftar_produk
]

# Cari produk sales yang tidak ada dalam TXT
produk_tidak_dikenal = sorted(
    set(sales["nama_produk"]) - set(daftar_produk)
)

print(
    "Jumlah nama produk yang tidak ditemukan:",
    len(produk_tidak_dikenal)
)

for produk in produk_tidak_dikenal:
    print(produk)

Jumlah nama produk yang tidak ditemukan: 0
